# Capítulo 5 — Processamento de Linguagem Natural para Inteligência

**Inteligência Artificial e Machine Learning Avançado para Defesa** — Prof. Cristiano Alves · Quantum Strategic AI

---

### 🎯 Objetivos do capítulo

Ao final deste notebook, você será capaz de:

1. **Executar** o pré-processamento de texto (tokenização, normalização, remoção de *stopwords*, radicalização e lematização) e compreender por que o texto exige preparação distinta da de imagens e dados tabulares;
2. **Representar** texto numericamente — das representações **esparsas** (*bag-of-words*, TF-IDF) às **densas** (*word embeddings*) — sabendo o que cada uma captura e o que descarta;
3. **Construir** classificadores supervisionados de texto para **análise de sentimento** e **classificação temática**, com métricas adequadas ao contexto operacional;
4. **Aplicar** PLN à **detecção de desinformação** e à identificação de **campanhas coordenadas**, combinando sinais linguísticos com sinais comportamentais — e reconhecer os limites e riscos dessa tarefa;
5. **Compreender** a tradução automática neural (*seq2seq* com **atenção**) e o seu papel operacional, e ver como a atenção conduz diretamente ao *Transformer* do Capítulo 6.

> A visão foi a primeira metade da percepção; a **linguagem** é a outra. E, em inteligência, é a metade **dominante**: a maior parte da informação chega em forma de texto — comunicações interceptadas, fontes abertas (OSINT), relatórios, redes sociais. O processamento de linguagem natural (PLN) é a disciplina que extrai desse texto **sentido operacional**: do que trata, que tom carrega, se é autêntico, o que afirma.
>
> Seguimos a mesma progressão dos módulos anteriores: primeiro as representações e os modelos clássicos, que formam a *baseline*; depois a virada neural, culminando no **mecanismo de atenção** que prepara o Capítulo 6. A disciplina não muda: comparar contra uma *baseline*, medir pelo custo operacional do erro e — em um domínio onde o erro pode significar **censura ou atribuição indevida** — preservar, mais do que nunca, o **julgamento humano**.

## Preparação do ambiente

Quase todo o capítulo executa com `numpy`, `matplotlib` e `scikit-learn`, já presentes no Colab — inclusive sobre um **corpus sintético em português**, gerado no próprio notebook (dados de fontes abertas reais não são necessários para o aprendizado das técnicas). Apenas a Seção 5.4 (tradução neural) usa a biblioteca `transformers`, instalada em célula própria.

Como sempre, fixamos a **semente aleatória**: *dados os mesmos dados, os mesmos resultados*.

In [ ]:
# Preparacao do ambiente: importacoes e semente de reprodutibilidade
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn

SEMENTE = 0
np.random.seed(SEMENTE)

print(f"numpy        {np.__version__}")
print(f"matplotlib   {matplotlib.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print("Ambiente pronto.")

---
## 5.1 Do texto ao vetor: pré-processamento e representação

Modelos operam sobre números; texto é uma sequência de símbolos. A primeira tarefa do PLN é, portanto, **converter texto em vetores** — e fazê-lo sem descartar o que importa.

### 5.1.1 Pré-processamento

Antes de vetorizar, normaliza-se o texto:

- A **tokenização** o quebra em unidades (palavras, subpalavras, pontuação);
- A **normalização** uniformiza maiúsculas, acentos e variações gráficas;
- A **remoção de *stopwords*** descarta palavras muito frequentes e pouco informativas (*de*, *a*, *que*);
- A **radicalização** (*stemming*) ou a **lematização** reduzem palavras à sua raiz ou forma canônica — *detectou*, *detectando* e *detectar* colapsam em um único termo.

O **português, de morfologia rica, torna a lematização particularmente útil**, e a escolha de cada etapa depende da tarefa: para classificação temática, agregar formas ajuda; para análise estilística, pode apagar sinais valiosos.

In [ ]:
# O pre-processamento passo a passo, sobre uma frase de relatorio
import re
import unicodedata

frase = ("As embarcações não identificadas foram detectadas ao amanhecer, "
         "deslocando-se em direção ao litoral norte.")

# 1) Tokenizacao: quebra em palavras
tokens = re.findall(r"\w+", frase)

# 2) Normalizacao: caixa baixa + remocao de acentos
def normaliza(t):
    t = t.lower()
    return unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()

normalizados = [normaliza(t) for t in tokens]

# 3) Remocao de stopwords (lista minima de portugues)
STOPWORDS = {"a", "o", "as", "os", "um", "uma", "de", "do", "da", "dos",
             "das", "em", "no", "na", "nos", "nas", "ao", "aos", "e", "que",
             "com", "por", "para", "se", "ser", "foi", "foram", "nao"}
sem_stop = [t for t in normalizados if t not in STOPWORDS]

# 4) Radicalizacao RUDIMENTAR (didatica): remove sufixos frequentes.
# Em producao, use um lematizador de portugues (ex.: spaCy pt_core_news).
def radical(t):
    for sufixo in ("acoes", "acao", "ando", "endo", "indo", "adas", "ados",
                   "ada", "ado", "ndo", "es", "s"):
        if t.endswith(sufixo) and len(t) - len(sufixo) >= 4:
            return t[: -len(sufixo)]
    return t

radicais = [radical(t) for t in sem_stop]

print("original:     ", frase)
print("tokens:       ", tokens[:8], "...")
print("normalizados: ", normalizados[:8], "...")
print("sem stopwords:", sem_stop)
print("radicais:     ", radicais)

**Observe:** *detectadas* virou `detect` — a mesma raiz de *detectou* e *detectar*: para um classificador temático, as três formas passam a **contar como o mesmo termo**. Note também o que se perdeu: a negação "*não* identificadas" sumiu com as *stopwords* — para análise de sentimento, isso pode inverter o sentido. **Cada etapa é uma decisão, não um ritual.**

### 5.1.2 Representações esparsas: bag-of-words e TF-IDF

A representação mais simples é o **saco de palavras** (*bag-of-words*): conta-se quantas vezes cada termo do vocabulário ocorre no documento, produzindo um vetor longo e esparso. Sua limitação é evidente — **ignora a ordem** ("o drone abateu o alvo" e "o alvo abateu o drone" recebem o mesmo vetor) e trata todos os termos como igualmente importantes.

O **TF-IDF** (*Term Frequency–Inverse Document Frequency*) corrige a segunda falha, ponderando cada termo por sua frequência no documento e por sua **raridade no corpus**:

$$\mathrm{tf\_idf}(t, d) = \mathrm{tf}(t, d) \cdot \log \frac{N}{\mathrm{df}(t)},$$

onde $\mathrm{tf}(t, d)$ é a frequência do termo $t$ no documento $d$, $N$ é o número de documentos e $\mathrm{df}(t)$ é o número de documentos que contêm $t$. Termos comuns a quase todos os textos (baixo poder discriminante) recebem **peso pequeno**; termos raros e característicos, **peso alto**. É uma representação robusta, barata e — combinada a um classificador linear (Capítulo 2) — uma ***baseline* de texto notavelmente forte**.

In [ ]:
# O TF-IDF calculado a mao, em um mini-corpus de 4 relatorios
mini_corpus = [
    "embarcacao detectada na area de patrulha",
    "comboio terrestre na area de fronteira",
    "embarcacao de pesca na area costeira",
    "subestacao de energia na area urbana",
]
N = len(mini_corpus)
docs_tokens = [d.split() for d in mini_corpus]

termos_interesse = ["area", "embarcacao", "subestacao"]
print(f"{'termo':14s} {'df(t)':>6s} {'idf = log(N/df)':>16s}   tf-idf por doc")
for t in termos_interesse:
    df = sum(t in tokens for tokens in docs_tokens)
    idf = np.log(N / df)
    tfidf_docs = [round(float(tokens.count(t) * idf), 3) for tokens in docs_tokens]
    print(f"{t:14s} {df:6d} {idf:16.3f}   {tfidf_docs}")

print("\n'area' aparece em TODOS os documentos: idf = log(4/4) = 0 --")
print("peso nulo em todos. O TF-IDF descobre sozinho as stopwords do dominio.")

**Observe:** o termo onipresente recebe **peso zero** — sem lista manual, o TF-IDF neutraliza as *stopwords do domínio* (o Exercício 5.2 formaliza esse cálculo).

### 5.1.3 Representações densas: word embeddings

As representações esparsas **ignoram o significado**: os vetores de *navio* e *embarcação* são tão distantes quanto os de *navio* e *nuvem*. As ***word embeddings*** (word2vec, GloVe) resolvem isso mapeando cada palavra a um **vetor denso** de dimensão modesta, aprendido de grandes corpora sob a **hipótese distribucional**: *palavras que ocorrem em contextos semelhantes têm significados semelhantes*. Os vetores resultantes capturam relações surpreendentes — a aritmética $\mathit{rei} - \mathit{homem} + \mathit{mulher} \approx \mathit{rainha}$ tornou-se o exemplo canônico.

A limitação das *embeddings* clássicas é serem **estáticas**: cada palavra tem um único vetor, independentemente do contexto — *manga* (fruta) e *manga* (da camisa) compartilham o mesmo vetor. Superá-la exige **representações contextuais**, em que o vetor depende da frase — precisamente o que os modelos baseados em *Transformer* (Capítulo 6) fornecem.

> **📝 Nota** — A progressão das representações de texto — *bag-of-words* → TF-IDF → *embeddings* estáticas → *embeddings* contextuais — é uma história de captura crescente de significado, **ao custo crescente de dados e de computação**. A escolha operacional não é "usar a mais avançada", mas **a mais simples que resolve a tarefa com garantias suficientes**. Para muitos problemas de classificação temática, o TF-IDF continua competitivo, mais barato e muito mais auditável do que uma rede profunda.

Para trabalhar, precisamos de um **corpus**. Como nos capítulos anteriores, geramos **dados sintéticos** que reproduzem as dificuldades reais — aqui, relatos curtos de fontes abertas em três temas, com **vocabulário deliberadamente compartilhado** entre eles (*comboio*, *tráfego* e *radar* aparecem em mais de um tema; é o contexto que desambigua):

In [ ]:
# Corpus sintetico de fontes abertas: 3 temas, vocabulario sobreposto
TEMAS = ["atividade maritima", "infraestrutura critica", "logistica terrestre"]

def gera_corpus_tematico(n_por_tema=200, semente=SEMENTE):
    rng = np.random.default_rng(semente)
    # objetos por tema: note os termos AMBIGUOS (comboio, trafego, radar,
    # manutencao) -- so o bigrama desambigua o tema.
    objetos = [
        ["comboio de embarcacoes", "trafego maritimo", "radar costeiro",
         "embarcacao de pesca", "lancha rapida", "navio mercante",
         "patrulha naval", "boia de sinalizacao"],
        ["rede eletrica", "subestacao de energia", "linha de transmissao",
         "manutencao da refinaria", "oleoduto principal", "torre de comunicacao",
         "estacao de tratamento", "gerador auxiliar"],
        ["comboio terrestre", "trafego rodoviario", "radar de estrada",
         "coluna de viaturas", "caminhao de suprimentos", "posto de bloqueio",
         "manutencao da ponte", "equipe de engenharia"],
    ]
    locais = [
        ["litoral norte", "faixa costeira", "canal de acesso", "porto velho",
         "ancoradouro", "ilha grande"],
        ["zona industrial", "perimetro urbano", "vale central",
         "distrito leste", "margem da represa", "complexo sul"],
        ["fronteira oeste", "rodovia federal", "serra alta", "vila do campo",
         "entroncamento", "passagem do rio"],
    ]
    modelos_frase = [
        "observado {obj} nas proximidades de {loc} durante a madrugada",
        "relatorio indica aumento de {obj} em {loc} nas ultimas horas",
        "fontes locais reportam {obj} proximo a {loc} sem identificacao",
        "registrada movimentacao de {obj} na regiao de {loc} ontem",
        "equipe confirma presenca de {obj} na area de {loc}",
        "imagens mostram {obj} deslocando-se para {loc} ao amanhecer",
    ]
    # pares ESPELHADOS entre os temas 0 e 2: exatamente os mesmos
    # unigramas, ordem (e tema!) diferentes -- so o bigrama desambigua.
    espelhados = [
        ("embarcacoes transportando viaturas para patrulha no setor {s}",
         "viaturas transportando embarcacoes para patrulha no setor {s}"),
        ("comboio de balsas com carga de caminhoes cruzou o corredor {s}",
         "comboio de caminhoes com carga de balsas cruzou o corredor {s}"),
    ]
    setores = ["norte", "sul", "leste", "oeste", "central"]
    n_espelhados = int(0.15 * n_por_tema)

    documentos, rotulos = [], []
    for tema in range(3):
        for _ in range(n_por_tema - (n_espelhados if tema != 1 else 0)):
            modelo = modelos_frase[rng.integers(len(modelos_frase))]
            doc = modelo.format(obj=objetos[tema][rng.integers(8)],
                                loc=locais[tema][rng.integers(6)])
            documentos.append(doc)
            rotulos.append(tema)
    for _ in range(n_espelhados):
        par = espelhados[rng.integers(len(espelhados))]
        s = setores[rng.integers(len(setores))]
        documentos.append(par[0].format(s=s)); rotulos.append(0)
        documentos.append(par[1].format(s=s)); rotulos.append(2)
    return documentos, np.array(rotulos)

documentos, rotulos = gera_corpus_tematico()
print(f"{len(documentos)} documentos, 3 temas: {TEMAS}\n")
rng = np.random.default_rng(SEMENTE)
for i in rng.choice(len(documentos), 6, replace=False):
    print(f"[{TEMAS[rotulos[i]][:22]:22s}] {documentos[i]}")

In [ ]:
# A hipotese distribucional em acao: embeddings por co-ocorrencia + SVD
# (a mesma ideia por tras de word2vec/GloVe, em versao minima e auditavel)
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# matriz termo-documento -> co-ocorrencia termo-termo -> SVD (densa)
contador = CountVectorizer(min_df=3)
X_contagem = contador.fit_transform(documentos)
vocab = contador.get_feature_names_out()
C = (X_contagem.T @ X_contagem).toarray().astype(float)   # co-ocorrencia
np.fill_diagonal(C, 0.0)
C = np.log1p(C)                                           # amortece contagens

svd = TruncatedSVD(n_components=16, random_state=SEMENTE)
vetores = svd.fit_transform(C)                            # 1 vetor por termo
indice = {t: k for k, t in enumerate(vocab)}

def similaridade(a, b):
    return float(cosine_similarity(vetores[indice[a]:indice[a] + 1],
                                   vetores[indice[b]:indice[b] + 1])[0, 0])

print("similaridade de cosseno entre vetores de termos:")
for a, b in [("pesca", "lancha"), ("pesca", "eletrica"),
             ("subestacao", "eletrica"), ("subestacao", "pesca"),
             ("navio", "mercante"), ("navio", "oleoduto")]:
    print(f"  {a:12s} ~ {b:12s} = {similaridade(a, b):+.2f}")

# projecao 2D dos termos mais frequentes, colorida pelo tema dominante
freq = np.asarray(X_contagem.sum(axis=0)).ravel()
top = np.argsort(freq)[::-1][:40]
proj = TruncatedSVD(n_components=2, random_state=SEMENTE).fit_transform(C)
X_por_tema = [np.asarray(X_contagem[rotulos == t].sum(axis=0)).ravel()
              for t in range(3)]
tema_dom = np.argmax(np.vstack(X_por_tema), axis=0)

plt.figure(figsize=(9, 6))
cores = ["tab:blue", "tab:orange", "tab:green"]
for k in top:
    plt.scatter(proj[k, 0], proj[k, 1], c=cores[tema_dom[k]], s=28)
    plt.annotate(vocab[k], (proj[k, 0], proj[k, 1]), fontsize=8.5,
                 xytext=(3, 3), textcoords="offset points")
for t, cor in enumerate(cores):
    plt.scatter([], [], c=cor, label=TEMAS[t])
plt.title("Termos que ocorrem em contextos semelhantes ficam proximos")
plt.legend(); plt.tight_layout(); plt.show()

**Observe:** sem nenhum rótulo, apenas observando **com que outras palavras cada termo co-ocorre**, os vetores agrupam o vocabulário por tema — *embarcações* fica perto de *pesca* e longe de *subestação*. É a hipótese distribucional em ação, e é exatamente o princípio (em escala milhões de vezes maior) de word2vec e GloVe. Note a limitação **estática**: *comboio* recebe **um único vetor**, a meio caminho entre o mar e a estrada — só uma representação **contextual** (Capítulo 6) o desambiguaria frase a frase.

---
## 5.2 Classificação de texto: análise de sentimento e classificação temática

A tarefa mais comum do PLN operacional é a **classificação de texto**: atribuir a cada documento um rótulo. A receita geral é a do Capítulo 2 — *texto → vetores → classificador* —, e nela reaparecem todas as suas lições, sobretudo as **métricas para classes desbalanceadas**.

A **classificação temática** roteia documentos por assunto: triar um volume de relatórios ou de material de fontes abertas por tema de interesse é, funcionalmente, o análogo textual da triagem de imagens do Capítulo 4. A **análise de sentimento** atribui polaridade (positivo, negativo, neutro) e serve para aferir o humor de uma população, monitorar narrativas ou apoiar a proteção da força. Ambas enfrentam dificuldades próprias do texto: **ironia e sarcasmo**, mistura de idiomas (*code-switching*), gírias e jargão de domínio.

> **🛡️ Contexto de defesa** — O monitoramento de fontes abertas enfrenta o mesmo problema de **volume** da vigilância por imagem: há mais texto público — notícias, redes, fóruns — do que qualquer equipe consegue ler. A classificação de texto é o **multiplicador de capacidade** que prioriza a fração relevante para a atenção humana. E vale a mesma ressalva de **soberania** do Capítulo 1: modelos e dados linguísticos de origem estrangeira carregam vieses e dependências que, em uso soberano, precisam ser ponderados.

A **Listagem 5.1** constrói a *baseline* de classificação temática — TF-IDF mais regressão logística. O passo neural, quando se justifica, substitui o TF-IDF por uma camada de *embedding* seguida de uma rede (uma CNN ou LSTM sobre texto, do Capítulo 3) — e deve, como sempre, **provar sua vantagem sobre a baseline**:

In [ ]:
# Listagem 5.1 - Baseline de classificacao tematica: TF-IDF e regressao
# logistica.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# documentos: lista de textos; rotulos: tema de cada texto (0, 1, 2)
X_tr, X_te, y_tr, y_te = train_test_split(
    documentos, rotulos, test_size=0.25, stratify=rotulos,
    random_state=SEMENTE)

# TF-IDF pondera cada termo pela frequencia no documento e pela raridade
# no corpus. n-gramas de 1 a 2 capturam expressoes curtas ("alvo movel").
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3,
                              sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

baseline.fit(X_tr, y_tr)
# Relatorio por classe: precisao, revocacao e F1 (ver Cap. 2).
print(classification_report(y_te, baseline.predict(X_te),
                            target_names=TEMAS, digits=3))

**Observe:** a *baseline* clássica resolve com folga a triagem temática — inclusive os documentos cujo único sinal de tema é um **bigrama** ("comboio *de embarcações*" vs. "comboio *terrestre*"); o Exercício 5.1 mede exatamente essa contribuição.

A **análise de sentimento** segue a mesma receita — muda apenas o rótulo. Treinamos abaixo um classificador de polaridade sobre relatos sintéticos de percepção pública (ele será um dos blocos do pipeline OSINT da Seção 5.5):

In [ ]:
# Analise de sentimento: a mesma receita, outro rotulo
POLARIDADES = ["negativo", "neutro", "positivo"]

def gera_corpus_sentimento(n_por_classe=150, semente=SEMENTE):
    rng = np.random.default_rng(semente)
    aberturas = {
        0: ["populacao relata insatisfacao com", "moradores denunciam falhas em",
            "cresce a revolta contra", "protestos apontam abandono de",
            "usuarios reclamam da precariedade de"],
        1: ["relatorio descreve situacao de", "boletim informa estado de",
            "nota tecnica registra dados de", "orgao divulga levantamento sobre",
            "comunicado atualiza informacoes de"],
        2: ["populacao elogia melhorias em", "moradores comemoram avancos em",
            "pesquisa aponta satisfacao com", "usuarios destacam qualidade de",
            "nota reconhece bons resultados de"],
    }
    servicos = ["abastecimento de agua", "fornecimento de energia",
                "transporte publico", "seguranca na regiao",
                "atendimento de saude", "rede de comunicacao",
                "iluminacao das vias", "coleta na cidade"]
    fechos = {
        0: ["apos semanas sem solucao", "com riscos para a comunidade",
            "em meio a sucessivas interrupcoes", "sem resposta das autoridades"],
        1: ["no ultimo trimestre", "conforme dados oficiais",
            "segundo o levantamento", "no periodo avaliado"],
        2: ["apos a conclusao das obras", "com resultados acima do esperado",
            "gracas ao novo plano", "desde a modernizacao do sistema"],
    }
    textos, polos = [], []
    for classe in range(3):
        for _ in range(n_por_classe):
            textos.append(f"{aberturas[classe][rng.integers(5)]} "
                          f"{servicos[rng.integers(8)]} "
                          f"{fechos[classe][rng.integers(4)]}")
            polos.append(classe)
    polos = np.array(polos)
    # ~8% de ruido de rotulagem: em sentimento, anotadores discordam --
    # e nenhum relatorio honesto de texto real exibe F1 = 1.000.
    n_ruido = int(0.08 * len(polos))
    idx = rng.choice(len(polos), n_ruido, replace=False)
    polos[idx] = rng.integers(0, 3, n_ruido)
    return textos, polos

textos_sent, polos = gera_corpus_sentimento()
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(
    textos_sent, polos, test_size=0.25, stratify=polos, random_state=SEMENTE)

modelo_sentimento = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
modelo_sentimento.fit(Xs_tr, ys_tr)
print(classification_report(ys_te, modelo_sentimento.predict(Xs_te),
                            target_names=POLARIDADES, digits=3))

---
## 5.3 Detecção de desinformação e campanhas coordenadas

Entre as aplicações de PLN em defesa, poucas são tão relevantes — e tão **delicadas** — quanto a detecção de desinformação e de campanhas coordenadas. Convém separar **dois problemas distintos**, de dificuldades muito diferentes.

### 5.3.1 O nível do conteúdo

O primeiro é classificar um texto como *provável desinformação*. É tentador tratá-lo como mais um problema de classificação, mas ele é **notavelmente difícil e frágil**. A veracidade raramente está na superfície linguística: um texto falso pode ser gramaticalmente impecável, e um verdadeiro, desajeitado. Classificadores de conteúdo tendem a captar **pistas superficiais** — estilo, fonte, palavras carregadas — que se correlacionam com desinformação nos dados de treino, mas que **um adversário adapta com facilidade**. A verificação factual genuína exige confrontar afirmações com **evidência externa**, tarefa que excede a classificação de texto.

### 5.3.2 O nível do comportamento

O segundo problema é mais tratável e, muitas vezes, mais informativo: identificar **coordenação**. Aqui a pergunta não é *o que* se diz, mas o **padrão de quem diz, quando e como**. Campanhas coordenadas deixam **assinaturas comportamentais** — conteúdo quase idêntico replicado por muitas contas, sincronização temporal de publicações, redes de amplificação. Esses sinais, combinando PLN com análise de redes e de séries temporais, são mais robustos do que a classificação de conteúdo, porque descrevem a **mecânica da campanha, não o seu texto**.

> **⚠️ Armadilha comum** — **Confundir um sinal de coordenação com prova de desinformação.** Conteúdo quase-duplicado indica *amplificação*, não *falsidade* — uma notícia verdadeira também é replicada. E classificadores de conteúdo, treinados em pistas superficiais, marcam como suspeito o discurso legítimo que apenas *se parece* com desinformação. O custo do falso positivo é aqui **excepcionalmente alto**: atribuição indevida, silenciamento de vozes legítimas, erro de consequências políticas. Nesse domínio, a saída do modelo é uma **hipótese para investigação humana, jamais um veredito** — e a decisão de rotular algo como desinformação **não pode ser automatizada**.

A **Listagem 5.2** ilustra um sinal simples e útil: a **amplificação por conteúdo quase-duplicado** entre autores distintos:

In [ ]:
# Listagem 5.2 - Sinal de coordenacao: amplificacao por conteudo
# quase-duplicado.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def deteccao_coordenacao(posts, autores, limiar_sim=0.85):
    # Sinaliza AMPLIFICACAO: mensagens muito semelhantes publicadas por
    # autores distintos -- indicio comportamental de coordenacao, NAO
    # uma prova. A similaridade e medida pelo cosseno dos vetores TF-IDF.
    vet = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    M = vet.fit_transform(posts)
    S = cosine_similarity(M)              # similaridade par a par

    suspeitos = []
    n = len(posts)
    for i in range(n):
        for j in range(i + 1, n):
            if S[i, j] >= limiar_sim and autores[i] != autores[j]:
                suspeitos.append((i, j, float(S[i, j])))

    # conjunto de autores envolvidos em conteudo quase-identico
    envolvidos = {autores[i] for i, j, _ in suspeitos}
    envolvidos |= {autores[j] for i, j, _ in suspeitos}
    return suspeitos, envolvidos

print("Funcao deteccao_coordenacao() definida.")

In [ ]:
# O sinal em acao: um pequeno feed com amplificacao embutida
posts = [
    # campanha coordenada: quase-duplicatas entre autores DISTINTOS
    "urgente: falta de energia atinge toda a zona norte ha tres dias e nada e feito pelas autoridades",
    "URGENTE - falta de energia atinge toda a zona norte ha 3 dias e nada e feito pelas autoridades!!",
    "urgente falta de energia atinge a zona norte ha tres dias e nada foi feito pelas autoridades",
    # replicacao legitima: MESMO autor repostando o proprio conteudo
    "novo boletim meteorologico indica chuvas fortes no litoral durante o fim de semana com ventos de sul",
    "novo boletim meteorologico indica chuvas fortes no litoral durante o fim de semana com ventos de sul",
    # conversa organica: mesmo assunto, redacoes independentes
    "minha rua na zona norte esta ha dias sem luz, alguem sabe quando a energia volta por aqui",
    "terceiro apagao do mes no meu bairro, a rede da zona norte precisa de manutencao seria e rapida",
    "o comercio da zona norte teve prejuizo com a falta de luz na semana passada, dizem os lojistas",
    # ruido nao relacionado
    "movimento normal de embarcacoes no porto nesta manha de terca, sem ocorrencias registradas na area",
    "pescadores relatam boa manha no porto, com desembarque tranquilo e movimento dentro do esperado",
    "obras da nova subestacao devem comecar no proximo semestre, informa nota da prefeitura desta terca",
    "cronograma de obras do viaduto foi divulgado pela prefeitura, com interdicao parcial no semestre",
    "feira de tecnologia abre inscricoes para expositores nacionais em setembro, informa a organizacao",
    "museu recebe em outubro exposicao sobre a historia do porto, com visitas guiadas pela manha",
]
autores = ["conta_a1", "conta_b2", "conta_c3", "boletim_met", "boletim_met",
           "morador_x", "morador_y", "lojista_k", "obs_porto", "pescador_z",
           "imprensa_w", "imprensa_q", "agenda_t", "agenda_u"]

suspeitos, envolvidos = deteccao_coordenacao(posts, autores)
print("pares quase-duplicados entre autores distintos:")
for i, j, s in suspeitos:
    print(f"  ({i}, {j}) sim={s:.2f}: {autores[i]} ~ {autores[j]}")
print(f"\nautores envolvidos: {sorted(envolvidos)}")

print("\nefeito do limiar de similaridade:")
for limiar in [0.70, 0.80, 0.90, 0.95]:
    s, e = deteccao_coordenacao(posts, autores, limiar_sim=limiar)
    print(f"  limiar {limiar:.2f} -> {len(s)} pares, {len(e)} autores")

**Observe:** o sinal captura o trio de contas replicando o mesmo texto — e, corretamente, **ignora** a repostagem do mesmo autor e a conversa orgânica sobre o mesmo assunto (redações independentes têm similaridade baixa). Note o que o sinal **não diz**: o conteúdo amplificado pode até ser *verdadeiro* — o indício é de **coordenação**, e vira ponto de partida de investigação humana, nunca veredito.

---
## 5.4 Tradução automática neural

A última peça do PLN clássico-a-neural é a **tradução automática**, e é ela que nos leva à porta do Capítulo 6. Traduzir é um problema de **sequência para sequência** (*seq2seq*): a entrada e a saída têm comprimentos variáveis e não alinhados.

### 5.4.1 Codificador-decodificador

A arquitetura neural clássica para *seq2seq* é o **codificador-decodificador**. Um codificador (uma LSTM, do Capítulo 3) lê a frase de entrada e a resume em um **vetor de contexto**; um decodificador gera, a partir desse vetor, a frase de saída, palavra a palavra. O problema é o **gargalo**: comprimir uma frase longa inteira em um único vetor de tamanho fixo perde informação — e a qualidade despenca com o comprimento.

### 5.4.2 Atenção

A solução foi o **mecanismo de atenção**, e sua importância transcende a tradução. Em vez de um único vetor de contexto, o decodificador, a cada palavra que gera, **consulta toda a entrada** e pondera as partes relevantes. O contexto para o passo $i$ torna-se uma soma ponderada dos estados do codificador:

$$\mathbf{c}_i = \sum_j \alpha_{ij}\, \mathbf{h}_j, \qquad \alpha_{ij} = \frac{\exp(e_{ij})}{\sum_k \exp(e_{ik})},$$

onde $\mathbf{h}_j$ é o estado do codificador na posição $j$ e $\alpha_{ij}$ é o **peso de atenção** — o quanto a saída na posição $i$ deve "olhar" para a entrada na posição $j$ —, obtido por *softmax* sobre escores de compatibilidade $e_{ij}$. A atenção resolveu o gargalo e, ao fazê-lo, revelou uma ideia mais profunda: **talvez a recorrência seja dispensável e a atenção, sozinha, baste**. É essa a tese do *Transformer*, tema do Capítulo 6.

A célula abaixo mostra o mecanismo em sua forma mínima, usando os vetores distribucionais da Seção 5.1: uma **consulta** pondera os termos de uma frase pela compatibilidade — e o contexto resultante é a soma ponderada:

In [ ]:
# O mecanismo de atencao em forma minima (com os vetores da Secao 5.1)
def atencao(consulta, chaves, temperatura=0.15):
    # escores de compatibilidade -> softmax -> pesos que somam 1
    escores = np.array([float(cosine_similarity(consulta[None], c[None])[0, 0])
                        for c in chaves])
    pesos = np.exp(escores / temperatura)
    pesos = pesos / pesos.sum()
    contexto = (pesos[:, None] * chaves).sum(axis=0)
    return pesos, contexto

frase = ("registrada movimentacao de embarcacao de pesca e caminhao de "
         "suprimentos na area do porto").split()
FUNCIONAIS = {"de", "do", "da", "na", "e", "o", "a"}
presentes = [t for t in frase if t in indice and t not in FUNCIONAIS]
chaves = np.array([vetores[indice[t]] for t in presentes])

fig, eixos = plt.subplots(1, 2, figsize=(12.5, 3.8), sharey=True)
for eixo, termo_consulta in zip(eixos, ["pesca", "estrada"]):
    pesos, contexto = atencao(vetores[indice[termo_consulta]], chaves)
    eixo.bar(range(len(presentes)), pesos, color="tab:blue")
    eixo.set_xticks(range(len(presentes)))
    eixo.set_xticklabels(presentes, rotation=45, ha="right", fontsize=9)
    eixo.set_title(f"consulta: '{termo_consulta}'")
eixos[0].set_ylabel("peso de atencao")
plt.suptitle("A MESMA frase, ponderada de formas diferentes por consulta")
plt.tight_layout(); plt.show()

print("A atencao dissolve o gargalo: em vez de UM vetor fixo para a frase")
print("inteira, cada passo da saida consulta a entrada e pondera o que importa.")

**Observe:** a **mesma frase** produz contextos diferentes conforme a consulta — *costeira* concentra os pesos em *embarcação/pesca/porto*; *rodovia*, em *caminhão/suprimentos*. No *seq2seq* real, os escores de compatibilidade são **aprendidos** (não fixados pelo cosseno), mas o mecanismo é exatamente este.

A **Listagem 5.3** usa um modelo de tradução neural **pré-treinado** (família Marian/OPUS-MT) via biblioteca `transformers`. A célula seguinte instala a **série 4.x** da biblioteca (é a da API usada no livro); o modelo, ~310 MB, é baixado na primeira execução:

In [ ]:
# A API pipeline("translation") usada no livro e a da serie 4.x da
# biblioteca transformers (a serie 5 reorganizou as tasks de traducao).
%pip install -q "transformers<5" sentencepiece

In [ ]:
# Listagem 5.3 - Traducao automatica neural com modelo pre-treinado.
import os
os.environ["USE_TF"] = "0"     # backend PyTorch (evita conflito com Keras 3)
from transformers import pipeline

# Traducao neural com modelo pre-treinado (familia Marian / OPUS-MT).
# Ha modelos abertos para centenas de pares de idiomas.
tradutor = pipeline("translation",
                    model="Helsinki-NLP/opus-mt-en-ROMANCE")

# O modelo multi-alvo ROMANCE exige o token do idioma de destino:
texto = ">>pt_BR<< The convoy was detected moving north at first light."
saida = tradutor(texto, max_length=100)
print(saida[0]["translation_text"])

# ATENCAO operacional: um erro de traducao em material sensivel pode
# inverter o sentido de uma mensagem. A traducao automatica TRIA e
# apoia; a validacao por um linguista permanece indispensavel em
# decisoes de alto risco.

**Observe:** dentro desse tradutor está exatamente a arquitetura desta seção — codificador, decodificador e **atenção** entre eles. O aviso do rodapé não é retórico: em material sensível, um erro de tradução pode **inverter o sentido** de uma mensagem (o Exercício 5.6 constrói um caso). A tradução automática **tria e apoia**; a validação por linguista permanece indispensável em decisões de alto risco.

---
## 5.5 Aplicação integrada: triagem de fontes abertas (OSINT)

Reunimos as peças em um **fluxo de triagem OSINT**. Um volume contínuo de texto de fontes abertas precisa ser filtrado por **tema** e relevância, ter seu **sentimento** aferido e ser examinado quanto a sinais de **coordenação** — tudo para **priorizar a atenção do analista, não para substituí-la**.

A **Listagem 5.4** compõe os blocos deste capítulo: a classificação temática da Listagem 5.1, o modelo de sentimento da Seção 5.2 e o sinal de coordenação da Listagem 5.2.

> **📝 Nota** — No livro, o sentimento vem de um modelo BERT multilíngue pré-treinado (`nlptown/bert-base-multilingual-uncased-sentiment`, ~670 MB). Aqui usamos o classificador de sentimento **treinado na Seção 5.2** — mais leve, reproduzível e suficiente para o fluxo didático; a chamada original fica indicada em comentário. Em produção, vale a regra de sempre: o modelo maior tem de **provar sua vantagem** sobre esta *baseline*.

In [ ]:
# Listagem 5.4 (adaptada) - Pipeline de triagem OSINT: tema, sentimento
# e coordenacao.

# Fluxo simulado de fontes abertas: (texto, autor) -- inclui itens
# relevantes, ruido e uma campanha de amplificacao embutida.
fluxo = [
    ("fontes locais reportam comboio de embarcacoes proximo ao litoral norte sem identificacao", "obs_31"),
    ("populacao relata insatisfacao com seguranca na regiao do porto apos sucessivas interrupcoes", "morador_r"),
    ("urgente: embarcacoes estrangeiras invadem aguas do litoral norte, autoridades omissas", "conta_a1"),
    ("URGENTE!! embarcacoes estrangeiras invadem as aguas do litoral norte e autoridades omissas", "conta_b2"),
    ("urgente embarcacoes estrangeiras invadem aguas do litoral norte autoridades sao omissas", "conta_c3"),
    ("registrada movimentacao de caminhao de suprimentos na rodovia federal ontem", "obs_17"),
    ("nota tecnica registra dados de fornecimento de energia no ultimo trimestre", "orgao_e"),
    ("populacao elogia melhorias em iluminacao das vias apos a conclusao das obras", "morador_s"),
    ("imagens mostram lancha rapida deslocando-se para o canal de acesso ao amanhecer", "obs_09"),
    ("moradores denunciam falhas em rede de comunicacao na faixa costeira sem resposta das autoridades", "morador_t"),
]
fluxo_textos = [t for t, _ in fluxo]
fluxo_autores = [a for _, a in fluxo]

# 1. Classificacao tematica: reusa a baseline da Listagem 5.1.
temas = baseline.predict(fluxo_textos)

# 2. Sentimento: classificador treinado na Secao 5.2.
#    (No livro: pipeline("sentiment-analysis",
#         model="nlptown/bert-base-multilingual-uncased-sentiment"))
polaridade = modelo_sentimento.predict(fluxo_textos)

# 3. Sinal de coordenacao: reusa a Listagem 5.2.
_, autores_suspeitos = deteccao_coordenacao(fluxo_textos, fluxo_autores)

# 4. Fila priorizada: itens do tema prioritario que sejam de sentimento
#    muito negativo OU ligados a amplificacao coordenada.
TEMAS_PRIORITARIOS = {0}                     # atividade maritima
fila = []
for i, texto in enumerate(fluxo_textos):
    relevante = temas[i] in TEMAS_PRIORITARIOS
    coordenado = fluxo_autores[i] in autores_suspeitos
    negativo = polaridade[i] == 0            # classe "negativo"
    if relevante and (coordenado or negativo):
        fila.append({"idx": i, "tema": TEMAS[temas[i]],
                     "sentimento": POLARIDADES[polaridade[i]],
                     "coordenado": coordenado})

print(f"Itens sinalizados para revisao humana: {len(fila)}\n")
for item in fila:
    print(f"  [{'COORD' if item['coordenado'] else 'sent.'}] "
          f"{fluxo_textos[item['idx']][:72]}")
# A decisao final -- inclusive rotular algo como desinformacao -- e
# sempre humana (controle humano significativo, Cap. 10).

**Observe:** a fila entrega ao analista **poucos itens, com o motivo da sinalização** — tema prioritário amplificado de forma coordenada, ou relato negativo relevante — e deixa passar o ruído e o que é rotina. O padrão é, mais uma vez, o de **triagem sob supervisão humana** — mas em nenhum domínio deste livro a fronteira entre *apoiar* e *decidir* é tão sensível. Um classificador que rotule discurso legítimo como desinformação, ou um sinal de coordenação interpretado como prova, produz danos que nenhuma métrica de acurácia captura. A **humildade epistêmica** — tratar toda saída como hipótese sujeita a erro — e o **controle humano significativo** sobre a decisão não são, aqui, boas práticas: são **salvaguardas contra o abuso**, que o Capítulo 10 examinará como questão de ética e governança.

> **✏️ Experimente:** acrescente ao `fluxo` um relato *verdadeiro e legítimo* sobre embarcações que, por coincidência de redação, se pareça com os posts da campanha — e veja-o entrar na fila. É o falso positivo em ação: por isso a fila é *entrada de investigação*, não lista de culpados.

---
## 5.6 O caminho à frente

Este capítulo percorreu o PLN aplicado à inteligência — do pré-processamento e das representações de texto à classificação, à detecção de coordenação e à tradução — e chegou ao **mecanismo de atenção**, que dissolveu o gargalo do *seq2seq*. Com ele, encerra-se a percepção clássica-a-neural do Módulo II.

O **Capítulo 6** recolhe a deixa da atenção e a leva à sua conclusão: a arquitetura ***Transformer***, que abandona a recorrência, e os **modelos de linguagem de grande escala** (BERT, GPT e variantes) construídos sobre ela — com suas aplicações em OSINT, sumarização e apoio à decisão, e seus riscos próprios de **alucinação**, **vazamento de dados** e **dependência de modelos estrangeiros**.

A disciplina segue firme e cumulativa: comparar contra uma *baseline*, medir pelo custo operacional do erro, validar nas condições reais de emprego e manter o humano no centro da decisão. Ao entrarmos no território dos grandes modelos de linguagem, ela se tornará não apenas necessária, mas **a própria condição de um emprego responsável**.

## 📋 Resumo do capítulo

- O PLN converte texto em vetores por um **pré-processamento** (tokenização, normalização, *stopwords*, lematização) seguido de **representação**. O português, de morfologia rica, torna a lematização especialmente útil.

- As representações **esparsas** (*bag-of-words*, TF-IDF) são baratas, robustas e auditáveis, mas ignoram ordem e significado. As ***embeddings* densas** capturam semântica (a hipótese distribucional), porém, na forma clássica, são **estáticas** — um vetor por palavra, sem contexto.

- A **classificação de texto** — temática e de sentimento — segue a receita do Capítulo 2 e é o multiplicador de capacidade que tria o volume de fontes abertas. O **TF-IDF com classificador linear é uma baseline forte**.

- Na desinformação, o nível do **conteúdo** é frágil e adaptável pelo adversário; o nível do **comportamento** — coordenação, amplificação por conteúdo quase-duplicado, sincronização — é mais robusto, mas fornece **indícios, não provas**.

- A **tradução automática** é um problema *seq2seq*; o codificador-decodificador sofre do **gargalo** do vetor de contexto único, que o **mecanismo de atenção** resolve ao ponderar toda a entrada a cada passo da saída.

- A atenção é a **semente do Transformer** (Capítulo 6). Em todo o capítulo, o alto custo do falso positivo — em especial na desinformação — impõe **humildade epistêmica** e **controle humano** sobre a decisão.

## ⚠️ Armadilhas comuns

1. **Adotar *embeddings* ou redes profundas por reflexo em classificação temática.** Para muitas tarefas, o TF-IDF com classificador linear iguala ou supera modelos densos, a custo menor e com maior auditabilidade. Exija que a alternativa neural prove sua vantagem.

2. **Ignorar o pré-processamento específico do idioma.** A morfologia do português e fenômenos como *code-switching*, ironia e jargão de domínio degradam modelos ajustados a outro registro. Valide sempre no domínio real de emprego.

3. **Tratar detecção de desinformação como simples classificação.** A veracidade não está na superfície do texto; classificadores captam pistas superficiais que o adversário contorna. Prefira sinais comportamentais e confronto com evidência externa.

4. **Confundir amplificação com falsidade.** Conteúdo quase-duplicado indica coordenação, não mentira — conteúdo verdadeiro também é replicado. O sinal é um ponto de partida de investigação, não um veredito.

5. **Confiar acriticamente em tradução automática de material sensível.** Um erro de tradução pode inverter o sentido de uma mensagem. Em decisões de alto risco, a validação por um linguista é indispensável.

6. **Automatizar a decisão em um domínio de alto custo de erro.** Rotular discurso legítimo como desinformação causa danos que a acurácia não mede. A saída do modelo é hipótese para revisão humana; a decisão de rótulo permanece humana (Capítulo 10).

---
## 🧭 Exercícios

Classificação: **Essencial** (fixação) · **Tático** (aplicação) · **Estratégico** (extensão criativa). Os exercícios de código têm células preparadas abaixo; os dissertativos podem ser respondidos em células de texto no próprio notebook.

### Essencial

**Exercício 5.1** — Execute a célula abaixo (Listagem 5.1 sobre o corpus temático). Em seguida, **remova os bigramas** (`ngram_range=(1, 1)`) e compare o **F1 macro** antes e depois. Em duas linhas, explique o que os bigramas acrescentam à representação.

In [ ]:
# Exercicio 5.1 - a contribuicao dos bigramas
from sklearn.metrics import f1_score

documentos_ex, rotulos_ex = gera_corpus_tematico()
Xe_tr, Xe_te, ye_tr, ye_te = train_test_split(
    documentos_ex, rotulos_ex, test_size=0.25, stratify=rotulos_ex,
    random_state=SEMENTE)

ngramas = (1, 2)          # <--- ALTERE AQUI para (1, 1) e reexecute
modelo_ex = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=ngramas, min_df=3,
                              sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
modelo_ex.fit(Xe_tr, ye_tr)
f1 = f1_score(ye_te, modelo_ex.predict(Xe_te), average="macro")
print(f"ngram_range={ngramas} -> F1 macro = {f1:.3f}")

# Sua resposta (2 linhas):
# Os bigramas acrescentam ... (pense em "comboio de embarcacoes" vs.
# "comboio terrestre": o que o unigrama "comboio" sozinho diz do tema?)

**Exercício 5.2** — Calcule, **à mão**, o TF-IDF de um termo que aparece em **todos** os documentos de um corpus. Mostre que seu peso IDF é nulo e explique, em uma frase, por que isso é desejável para uma *stopword de domínio*. Use a célula abaixo apenas para **conferir** o seu cálculo.

In [ ]:
# Exercicio 5.2 - conferencia do calculo a mao
N_docs = 4                # <--- ALTERE para o tamanho do seu corpus
df_termo = 4              # <--- termo presente em TODOS os documentos
tf_termo = 3              # <--- frequencia do termo no documento

idf = np.log(N_docs / df_termo)
print(f"idf = log({N_docs}/{df_termo}) = {idf:.3f}")
print(f"tf-idf = {tf_termo} x {idf:.3f} = {tf_termo * idf:.3f}")

# Sua resposta (1 frase):
# O peso nulo e desejavel porque um termo presente em todos os
# documentos ...

**Exercício 5.3** — Aplique a função `deteccao_coordenacao` da Listagem 5.2 a um pequeno conjunto de mensagens **que você mesmo construa**, incluindo dois pares quase-idênticos de autores distintos. Varie `limiar_sim` entre 0,7 e 0,95 e relate o efeito sobre o número de pares sinalizados.

In [ ]:
# Exercicio 5.3 - construa seu proprio cenario de amplificacao
meus_posts = [                    # <--- ALTERE: escreva suas mensagens,
    # incluindo dois pares quase-identicos de autores distintos
    "alerta maximo sobre o fechamento da ponte principal nesta sexta feira a noite",
    "ALERTA maximo sobre o fechamento da ponte principal nesta sexta a noite!!",
    "atencao motoristas: interdicao da avenida central no sabado pela manha",
    "avenida central em obras tera interdicao no sabado, atencao com os desvios da manha",
    "hoje o transito fluiu bem na regiao apesar da chuva que caiu no inicio do dia",
    "novo restaurante inaugura no bairro com cardapio regional e precos populares nesta semana",
]
meus_autores = ["a", "b", "c", "d", "e", "f"]   # <--- ALTERE se quiser

for limiar in [0.70, 0.80, 0.90, 0.95]:
    pares, envolvidos = deteccao_coordenacao(meus_posts, meus_autores,
                                             limiar_sim=limiar)
    print(f"limiar {limiar:.2f} -> {len(pares)} pares | "
          f"autores: {sorted(envolvidos)}")

# Sua resposta (poucas linhas):
# 1) Limiar baixo demais sinaliza ... ; alto demais deixa passar ...
# 2) O limiar adequado depende de ...

### Tático

**Exercício 5.4** — A célula abaixo substitui, na Listagem 5.1, a representação TF-IDF por ***embeddings*** (média dos vetores distribucionais das palavras do documento — Seção 5.1.3), mantendo o mesmo classificador. Compare o **F1 macro** das duas representações e discuta em que tipo de texto cada uma leva vantagem.

In [ ]:
# Exercicio 5.4 - TF-IDF vs. embeddings (media dos vetores do documento)
# (requer as celulas da Secao 5.1.3 executadas: `vetores` e `indice`)
def vetor_documento(texto):
    vs = [vetores[indice[t]] for t in texto.split() if t in indice]
    return np.mean(vs, axis=0) if vs else np.zeros(vetores.shape[1])

Xe_tr, Xe_te, ye_tr, ye_te = train_test_split(
    documentos, rotulos, test_size=0.25, stratify=rotulos,
    random_state=SEMENTE)

# representacao 1: TF-IDF (baseline da Listagem 5.1)
mod_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
mod_tfidf.fit(Xe_tr, ye_tr)
f1_tfidf = f1_score(ye_te, mod_tfidf.predict(Xe_te), average="macro")

# representacao 2: media das embeddings do documento
E_tr = np.vstack([vetor_documento(t) for t in Xe_tr])
E_te = np.vstack([vetor_documento(t) for t in Xe_te])
mod_emb = LogisticRegression(max_iter=1000,
                             class_weight="balanced").fit(E_tr, ye_tr)
f1_emb = f1_score(ye_te, mod_emb.predict(E_te), average="macro")

print(f"F1 macro TF-IDF:          {f1_tfidf:.3f}")
print(f"F1 macro embeddings-media: {f1_emb:.3f}")

# Sua resposta (poucas linhas):
# 1) Qual venceu neste corpus, e por que ...
# 2) Em que tipo de texto a media de embeddings levaria vantagem
#    (pense em vocabulario novo/sinonimos nao vistos no treino) ...

**Exercício 5.5** *(dissertativo)* — Projete, em **pseudocódigo**, um detector de campanha coordenada que combine **dois sinais**: a amplificação por conteúdo quase-duplicado (Listagem 5.2) e a **sincronização temporal** de publicações (muitas contas publicando na mesma janela de minutos). Explique por que a combinação de sinais é mais robusta do que qualquer um isolado.

*Responda em uma célula de texto abaixo.*

**Exercício 5.6** — Use a célula abaixo (Listagem 5.3) para traduzir **cinco frases curtas de domínio militar** e avalie manualmente a qualidade. Identifique um caso em que a tradução **altera o sentido operacional** e proponha uma salvaguarda de processo que o capturaria antes de uma decisão. *(Requer o tradutor da Seção 5.4 carregado.)*

In [ ]:
# Exercicio 5.6 - avaliacao manual de traducao em dominio militar
frases = [                    # <--- ALTERE: use frases do seu dominio
    "The convoy was detected moving north at first light.",
    "Hold fire until the target is positively identified.",
    "The unit will engage the objective at dawn.",
    "Friendly forces are holding the bridge on the east bank.",
    "Intelligence reports increased activity near the border.",
]
for f in frases:
    t = tradutor(">>pt_BR<< " + f, max_length=100)[0]["translation_text"]
    print(f"EN: {f}\nPT: {t}\n")

# Dica: repare na traducao de "Hold fire" -- o que ela manda fazer?

# Sua resposta (poucas linhas):
# 1) Caso em que o sentido operacional mudou (e o risco associado): ...
# 2) Salvaguarda de processo proposta: ...

### Estratégico

**Exercício 5.7** — Em um breve texto técnico (até duas páginas), discuta por que a detecção automática de desinformação **no nível do conteúdo** é epistemicamente frágil, e por que os **sinais comportamentais de coordenação** são mais defensáveis. Aborde explicitamente o risco de falsos positivos sobre discurso legítimo e os limites que ele impõe a qualquer emprego operacional.

**Exercício 5.8** — Projete, em diagrama, um **pipeline de triagem OSINT completo**, no espírito da Listagem 5.4, especificando: as fontes, as etapas de classificação e de detecção de coordenação, a métrica operacional, os **pontos de controle humano** na decisão e as salvaguardas contra a **atribuição indevida**. Justifique por que a decisão de rotular algo como desinformação **não é automatizada**.

**Exercício 5.9** — Tome um conjunto público de textos rotulados (por tema ou sentimento) e conduza um **estudo comparativo completo** entre a *baseline* TF-IDF e um modelo neural com *embeddings*, redigindo um miniartigo de até três páginas que documente: os dados, o pré-processamento, os resultados por classe com sua variância e uma recomendação fundamentada — incluindo os **vieses do corpus** e os limites de generalização para o registro linguístico real de emprego.

---

*Fim do Capítulo 5 — no Capítulo 6, a atenção levada à sua conclusão: o Transformer e os modelos de linguagem de grande escala.*